# M1B1 - Classification automatique des intentions clients - Modernisation d'un pipeline NLP chez Finova

## Étape 4 : Tester l'hypothèse du manager : embeddings figés

Vous avez désormais une baseline classique rigoureusement mesurée. Vous pouvez maintenant tester l'hypothèse du manager dans des conditions honnêtes : remplacer uniquement la couche features par des embeddings neuronaux, tout le reste identique, et voir si le gain est réel ou marketing.

   - Featurizer via `distilbert-base-uncased`.
   - Encoder train/val une seule fois (cache disque conseillé).
   - Réentraîner les mêmes classifieurs que les étapes 1-2 sur ces nouvelles features. Évaluer sur le même split 80/20, puis confirmer en CV comme à l'étape 3.
   - Ajouter chaque ligne au tableau comparatif.


### Featuring et Encodage des données

Installation des bibliothèques nécessaires à l'utilisation de distilbert-base-uncased :
`pip install transformers torch`

In [2]:
import numpy as np
from datetime import datetime
import time

# pour l'import des jeux de données
from datasets import load_dataset, concatenate_datasets

# pour l'embedding DistilBert
# from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModel
import torch

# Pour l'entrainement
from sklearn.linear_model import LogisticRegression

# pour la mise en cache
import os
import pickle

print(f"=====> Début éxécution : {datetime.now()}")

# ------------------------------
# CONFIG CACHE
# ------------------------------
CACHE_FILE = "embedding_cache_distilbert.pkl"

# charger cache existant si présent
if os.path.exists(CACHE_FILE):
    with open(CACHE_FILE, "rb") as f:
        cache = pickle.load(f)
    print("        ...Cache chargé :", len(cache), "éléments")
else:
    cache = {}
    print("        ...Cache initialisé")


# ------------------------------
# DATASET
# ----------------------------

# On charge le jeu de données Banking77
dataset = load_dataset("mteb/banking77")
train_data = dataset["train"]
test_data = dataset["test"]
# full_data = concatenate_datasets(dataset["train"], dataset["test"])

X = [str(t) for t in train_data["text"]]  # pour assurer que c'est bien du texte, sinon le tokenizer plante
y = train_data["label"]


# ------------------------------
# MODELE DISTILBERT
# ------------------------------

tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")
model = AutoModel.from_pretrained("distilbert-base-uncased")


# ------------------------------
# EMBEDDING
# ------------------------------

# embedding par batch pour contourner la limitation mémoire + mise en cache (disque)
def get_embeddings_batch(texts, batch_size=32):
    all_embeddings = []

    texts_to_compute = []
    indices_to_compute = []

    start_time = time.time()  # Temps de début
    
    # 1. Identifier les textes manquants dans le cache
    for i, text in enumerate(texts):
        if text not in cache:
            texts_to_compute.append(text)
            indices_to_compute.append(i)
    print(f"        ...Nouveaux textes à encoder : {len(texts_to_compute)}")

    # 2. Calcul en batch des embeddings manquants
    for i in range(0, len(texts_to_compute), batch_size):
        batch = texts_to_compute[i:i + batch_size]
        
        inputs = tokenizer(
            batch,
            padding=True,
            truncation=True,
            return_tensors="pt",
            max_length=128
        )

        with torch.no_grad():
            outputs = model(**inputs)

        embeddings = outputs.last_hidden_state.mean(dim=1)

        for text, emb in zip(batch, embeddings):
            cache[text] = emb.numpy()

    # 3. Sauvegarde du cache
    with open(CACHE_FILE, "wb") as f:
        pickle.dump(cache, f)

    print("        ...Cache mis à jour :", len(cache), "éléments")

    # 4. Reconstruction du dataset final
    for text in texts:
        all_embeddings.append(cache[text])

    return np.vstack(all_embeddings)

    elapsed = time.time() - start_time  # Durée
        
    #all_embeddings.append(embeddings)
    #return torch.cat(all_embeddings, dim=0)


# ------------------------------
# EXECUTION
# ------------------------------
print(f"=====> Début enbedding : {datetime.now()}")
X_embeddings = get_embeddings_batch(X)
# X_embeddings = get_embeddings_batch(texts).numpy()
# X_embeddings = get_embeddings(texts)


# ------------------------------
# ENTRAINEMENT
# ------------------------------
# Définition et entrainement du Modéle
print(f"=====> Début entrainement : {datetime.now()}")
model_lr = LogisticRegression(max_iter=1000)
model_lr.fit(X_embeddings, y)

print(f"=====> Fin éxécution : {datetime.now()}")


=====> Début éxécution : 2026-06-05 17:13:18.320560
        ...Cache chargé : 9993 éléments


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_transform.bias    | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


=====> Début enbedding : 2026-06-05 17:13:24.504852
        ...Nouveaux textes à encoder : 0
        ...Cache mis à jour : 9993 éléments
=====> Début entrainement : 2026-06-05 17:13:24.755818
=====> Fin éxécution : 2026-06-05 17:13:51.885148


### Entrainement des modèles

In [3]:
import pandas as pd
import numpy as np
import time
from tqdm.notebook import tqdm # pour afficher des barres de progression

# fonction permettant d'effectuer la validation croisée
from sklearn.model_selection import cross_val_score
# from sklearn.model_selection import KFold

from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import CountVectorizer, TfidfTransformer

# les modèles à comparer
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier

from sklearn.metrics import accuracy_score, accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, roc_auc_score


classifiers = {
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=42),
    "SVM": SVC(random_state=42)
    #"Random Forest": RandomForestClassifier(random_state=42),
    #"Naive Bayes": MultinomialNB()
    # "Decision Tree": DecisionTreeClassifier(random_state=42),
    # "KNN": KNeighborsClassifier(),
    # "Gradient Boosting": GradientBoostingClassifier(random_state=42),
    # "Naive Bayes": GaussianNB(),
    # "Neural Network": MLPClassifier(max_iter=1000, random_state=42),
}

results = []

# Itération sur la liste des classifier à tester
for name, clf in tqdm(classifiers.items(), desc="Progression"):
    start_time = time.time()  # Temps de début
        
    # Effectuer une validation croisée à 5 folds
    scores = cross_val_score(clf, X_embeddings, y, cv=5)
    accuracy = cross_val_score(clf, X_embeddings, y, cv=5, scoring='accuracy')
    f1_macro = cross_val_score(clf, X_embeddings, y, cv=5, scoring='f1_macro')
    f1_weighted = cross_val_score(clf, X_embeddings, y, cv=5, scoring='f1_weighted')
    # Calculer la moyenne des scores
    mean_score = scores.mean()
    
    # Calculer l'écart type des scores
    std_dev = scores.std()

    elapsed = time.time() - start_time  # Durée
    
    results.append({
            "Classifier": name,
            "Accuracy": accuracy.mean(),
            "F1-Macro": f1_macro.mean(),
            "F1-Weighted":  f1_weighted.mean(),
            "Durée Exec": round(elapsed, 2)
    })

# Affiche le temps dans la barre de progression
tqdm.write(f"Itération '{name}' exécutée en {elapsed:.2f} secondes")

# Afficher les résultats sous forme de tableau
results_df = pd.DataFrame(results)
# print(results_df.sort_values(by="F1-Score", ascending=False))
print(results_df.sort_values(by="F1-Weighted", ascending=False))

Progression:   0%|          | 0/2 [00:00<?, ?it/s]

Itération 'SVM' exécutée en 1147.78 secondes
            Classifier  Accuracy  F1-Macro  F1-Weighted  Durée Exec
0  Logistic Regression  0.877713  0.875794     0.876693      473.29
1                  SVM  0.828378  0.819388     0.826546     1147.78


## Étape 5 : Exposer le modèle : API FastAPI

Un modèle qui reste dans un notebook n'a aucune valeur métier. L'objectif final est de le rendre consommable par les autres équipes.

   - Exposer le meilleur modèle de l'étape 4. => le modèle choisi est `Logistic Regression`
   - Sérialiser via joblib.
   - Endpoints : POST /classify → {intent, confidence, model} ; GET /health ; GET /model-info (métriques tirées du tableau).
   - Tester via curl et /docs.

### Sérialisation via joblib

In [23]:
import joblib

# ------------------------------------------------------------------------------
# Chargement des données
# ------------------------------------------------------------------------------

# Conversion dans pandas, plus pratique - on ne prend que je jeu d'entrainement
df_train = dataset["train"].to_pandas()
df_test  = dataset["test"].to_pandas()

# On sépare le jeu de données, X = text 
X_train = df_train["text"]
y_train = df_train["label"]


# ------------------------------------------------------------------------------
# Sauvegarde
# ------------------------------------------------------------------------------

MODEL_PATH  = "SVCmodel.joblib"

text_clf = Pipeline([
    ('vect', CountVectorizer()),
    ('tfidf', TfidfTransformer()),
    ('clf', SVC(random_state=42)),  # le meilleur modèle
])

text_clf.fit(X, y)

joblib.dump(text_clf, MODEL_PATH) # génère le fichier model.joblib
        



['SVCmodel.joblib']

### Définition de l'API - Endpoints : POST /classify → {intent, confidence, model} 